# Bộ đánh giá Benchmark Suy luận Toán học cho LLM: Qwen2.5-Coder-7B-Instruct

Notebook thực hiện quy trình đánh giá chuẩn hóa năng lực suy luận toán học trên hai tập dữ liệu **MATH-500** và **GSM8K** với 4 phương pháp:
1. **Direct**: Zero-shot Direct Answering (Dự đoán đáp án trực tiếp).
2. **CoT**: Chain-of-Thought Step-by-Step Natural Language Reasoning (Suy luận từng bước bằng ngôn ngữ tự nhiên - Wei et al., NeurIPS 2022).
3. **SymCode**: Neurosymbolic Equation Solving với SymPy (ACL 2026) tích hợp vòng lặp tự sửa lỗi và bộ kiểm chứng toán học độc lập (Traceback + Independent Mathematical Verifier).
4. **SymPlanner**: Divide-and-Plan Neurosymbolic Program Synthesis (Stage 1 Divide -> Stage 2 Plan -> Stage 3 SymCode Execution -> Guarded Repair).

**Các tính năng nâng cao**:
- Cả hai tập dữ liệu **MATH-500** và **GSM8K** đều đã được phân loại chi tiết theo **Subject** và **Độ khó Level 1 đến Level 5**, cho phép lọc và đánh giá theo từng cấp độ mong muốn.
- Phân rã kết quả đánh giá đa chiều: Tổng thể (Overall), theo Chủ đề (by Subject), theo Mức độ khó (by Difficulty Level) và kết hợp Subject x Difficulty.
- Tự động lưu và khôi phục tiến trình (Auto-Checkpoint & Resume) giúp bảo vệ kết quả khi bị ngắt kết nối phiên làm việc trên Kaggle.
- Cơ chế In-Memory Fast Sandbox cách ly an toàn với tốc độ thực thi cao và bảo vệ chống treo tiến trình (Timeout).

In [ ]:
# 1. Xác định đường dẫn dự án và cài đặt môi trường
import os
import sys
import glob

# Cấu hình quản lý bộ nhớ PyTorch để tránh phân mảnh CUDA VRAM
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Tìm kiếm thư mục chứa package method trong môi trường Kaggle hoặc workspace cục bộ
def locate_directory(target_name):
    for search_root in ["/kaggle/input", ".", ".."]:
        if os.path.exists(search_root):
            for dirpath, dirnames, _ in os.walk(search_root):
                if target_name in dirnames:
                    return os.path.abspath(dirpath)
    return os.path.abspath(".")

ROOT_DIR = locate_directory("method")
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

print(f"[INFO] Thư mục gốc của dự án: {ROOT_DIR}")

# Tìm và cài đặt các thư viện phụ thuộc từ requirements.txt
req_matches = glob.glob("/kaggle/input/**/requirements.txt", recursive=True) or ["requirements.txt"]
req_file = req_matches[0] if (req_matches and os.path.exists(req_matches[0])) else os.path.join(ROOT_DIR, "requirements.txt")

if os.path.exists(req_file):
    os.system(f'pip install -q -r "{req_file}"')
else:
    os.system('pip install -q vllm bitsandbytes accelerate transformers sympy datasets tqdm')


In [ ]:
# 2. Nạp các module và kiểm tra thiết bị phần cứng
import gc
import io
import re
import math
import time
import json
import queue
import random
import traceback
import contextlib
import concurrent.futures
import torch
import pandas as pd

# Nạp trước các thư viện toán học biểu tượng
import sympy
import sympy as sp
import fractions
import itertools

from method import (
    LLMRunner,
    load_dataset_file,
    evaluate_direct_or_cot,
    evaluate_symcode,
    evaluate_symplanner,
    compute_metrics_table,
    save_benchmark_results,
    extract_boxed_content,
    extract_ground_truth,
    check_exact_match,
    verify_candidate_answer,
    execute_code_safely
)

print("[INFO] Các module benchmark đã sẵn sàng.")
print("Phiên bản PyTorch:", torch.__version__)
print("GPU Khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Tên GPU:", torch.cuda.get_device_name(0))
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Dung lượng VRAM tổng cộng: {vram_gb:.2f} GB")

In [ ]:
# 3. BẢNG ĐIỀU KHIỂN CẤU HÌNH BENCHMARK (CUSTOM CONFIGURATION)

def find_data_file(subpath_pattern):
    matches = glob.glob(f"/kaggle/input/**/{subpath_pattern}", recursive=True)
    if matches:
        return os.path.abspath(matches[0])
    local_p = os.path.join(ROOT_DIR, subpath_pattern)
    if os.path.exists(local_p):
        return os.path.abspath(local_p)
    if os.path.exists(subpath_pattern):
        return os.path.abspath(subpath_pattern)
    return subpath_pattern

# ==========================================================================
# BƯỚC 1: CHỌN TẬP DỮ LIỆU CẦN ĐÁNH GIÁ ("math500" hoặc "gsm8k")
# ==========================================================================
DATASET_CHOICE = "math500"  # Chọn "math500" hoặc "gsm8k"

# ==========================================================================
# BƯỚC 2: CHỌN CÁC PHƯƠNG PHÁP CẦN CHẠY
# Danh sách hỗ trợ: ["Direct", "CoT", "SymCode", "SymPlanner"]
# ==========================================================================
METHODS_TO_RUN = [
    "Direct",
    "CoT",
    "SymCode",
    "SymPlanner"
]

# ==========================================================================
# BƯỚC 3: CẤU HÌNH SỐ LƯỢNG MẪU VÀ BỘ LỌC MỨC ĐỘ KHÓ (LEVEL 1-5)
# Áp dụng linh hoạt cho cả MATH-500 và GSM8K:
# - num_samples: số nguyên (ví dụ: 5 để test nhanh) hoặc None (chạy toàn bộ tập dữ liệu)
# - filter_levels: [1, 2, 3] (chỉ lấy level mong muốn) hoặc None (lấy tất cả level 1-5)
# ==========================================================================
NUM_SAMPLES = 5  # Đặt thành None để chạy toàn bộ dữ liệu
FILTER_LEVELS = [1, 2, 3]  # Có thể chọn [1, 2], [1, 2, 3], [4, 5], hoặc None

# Tự động xác định đường dẫn file dữ liệu và tên file kết quả
if DATASET_CHOICE == "math500":
    dataset_path = find_data_file("data/math500/test.jsonl")
    hf_fallback = "HuggingFaceH4/MATH-500"
else:
    dataset_path = find_data_file("data/gsm8k/test.jsonl")
    hf_fallback = "openai/gsm8k"

if FILTER_LEVELS is not None and len(FILTER_LEVELS) > 0:
    lvl_str = "_lvl" + "_".join(str(lvl) for lvl in sorted(FILTER_LEVELS))
else:
    lvl_str = ""

out_name = f"{DATASET_CHOICE}{lvl_str}_results.json"
output_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
output_file = os.path.join(output_dir, out_name)

# Khi test nhanh <= 10 mẫu, xóa checkpoint cũ để đảm bảo kết quả sạch sẽ
if NUM_SAMPLES is not None and NUM_SAMPLES <= 10 and os.path.exists(output_file):
    os.remove(output_file)
    print(f"[INFO] Đã reset file kết quả test: {output_file}")

CONFIG = {
    "model_id": "Qwen/Qwen2.5-Coder-7B-Instruct",
    "use_vllm": False,
    "load_in_4bit": True,
    "max_new_tokens": 1024,
    "temperature": 0.0,
    "dataset_name": DATASET_CHOICE,
    "dataset_path": dataset_path,
    "hf_fallback": hf_fallback,
    "filter_levels": FILTER_LEVELS,
    "num_samples": NUM_SAMPLES,
    "methods_to_run": METHODS_TO_RUN,
    "code_exec_timeout": 15,
    "max_symcode_retries": 2,
    "output_file": output_file,
    "save_every": 1 if (NUM_SAMPLES and NUM_SAMPLES <= 10) else 5
}

print("=" * 70)
print(f"[INFO] Mô hình: {CONFIG['model_id']}")
print(f"[INFO] Tập dữ liệu: {CONFIG['dataset_name'].upper()} ({CONFIG['dataset_path']})")
print(f"[INFO] Mức độ lọc (Filter Levels): {CONFIG['filter_levels']}")
print(f"[INFO] Số mẫu đánh giá: {CONFIG['num_samples'] if CONFIG['num_samples'] is not None else 'TOÀN BỘ'}")
print(f"[INFO] Các phương pháp thực thi: {CONFIG['methods_to_run']}")
print(f"[INFO] File lưu kết quả: {CONFIG['output_file']}")
print("=" * 70)

In [ ]:
# 4. Dọn dẹp bộ nhớ VRAM, nạp Dataset và khởi tạo Model 4-bit
if "llm" in globals():
    del llm
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    current_mb = torch.cuda.memory_allocated() / (1024**2)
    print(f"[INFO] Bộ nhớ GPU hiện tại: {current_mb:.1f} MB")

dataset = load_dataset_file(
    CONFIG["dataset_path"],
    split="test",
    num_samples=CONFIG["num_samples"],
    filter_levels=CONFIG.get("filter_levels")
)

llm = LLMRunner(
    model_id=CONFIG["model_id"],
    use_vllm=CONFIG.get("use_vllm", True),
    load_in_4bit=CONFIG["load_in_4bit"],
    max_new_tokens=CONFIG["max_new_tokens"],
    temperature=CONFIG["temperature"]
)

In [ ]:
# 5. Thực thi các phương pháp đánh giá với cơ chế Auto-Resume
out_f = CONFIG["output_file"]
save_n = CONFIG.get("save_every", 5)

if os.path.exists(out_f):
    try:
        with open(out_f, "r", encoding="utf-8") as f:
            benchmark_data = json.load(f)
    except Exception:
        benchmark_data = {"config": CONFIG, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "results": {}, "summary": {}}
else:
    benchmark_data = {"config": CONFIG, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "results": {}, "summary": {}}

for method in CONFIG["methods_to_run"]:
    if method in ["Direct", "CoT"]:
        benchmark_data["results"][method] = evaluate_direct_or_cot(
            method, dataset, llm, checkpoint_file=out_f, save_every=save_n
        )
    elif method == "SymCode":
        benchmark_data["results"]["SymCode"] = evaluate_symcode(
            dataset, llm, timeout=CONFIG["code_exec_timeout"], max_retries=CONFIG["max_symcode_retries"], checkpoint_file=out_f, save_every=save_n
        )
    elif method == "SymPlanner":
        benchmark_data["results"]["SymPlanner"] = evaluate_symplanner(
            dataset, llm, timeout=CONFIG["code_exec_timeout"], max_retries=CONFIG["max_symcode_retries"], checkpoint_file=out_f, save_every=save_n
        )
    else:
        print(f"[WARN] Không nhận diện phương pháp: {method}. Bỏ qua...")

In [ ]:
# 6. Tổng hợp chỉ số và hiển thị bảng kết quả đa chiều
summary = compute_metrics_table(benchmark_data["results"])
benchmark_data["summary"] = summary
save_benchmark_results(benchmark_data, CONFIG["output_file"])

# Hiển thị bảng tổng thể (Overall Metrics)
df_overall = pd.DataFrame.from_dict({
    m: {
        "Accuracy (%)": v["accuracy_percent"],
        "Exact Match": f"{v['exact_match_count']}/{v['total_samples']}",
        "Avg Tokens": v["avg_generated_tokens"],
        "Avg Attempts": v["avg_attempts"],
        "Exec Success": v["execution_success_rate"],
        "Verif Pass": v["verification_success_rate"]
    }
    for m, v in summary.items()
}, orient="index")

print("")
print("--- BẢNG 1: TỔNG HỢP CHỈ SỐ OVERALL ---")
display(df_overall)

# Hiển thị bảng phân rã theo Chủ đề và Độ khó (Subject x Difficulty)
subj_diff_rows = []
for m, v in summary.items():
    for key, cell in v.get("by_subject_x_difficulty", {}).items():
        subj_diff_rows.append({
            "Method": m,
            "Subject": cell["subject"],
            "Difficulty": cell["difficulty"],
            "Correct": cell["correct"],
            "Total": cell["total"],
            "Accuracy (%)": cell["accuracy_percent"]
        })

if subj_diff_rows:
    print("")
    print("--- BẢNG 2: PHÂN RÃ ACCURACY THEO SUBJECT x DIFFICULTY ---")
    df_subj_diff = pd.DataFrame(subj_diff_rows)
    display(df_subj_diff)